# Natural Experiment — SES Covariate Balance Test

Tests the validity of the natural experiment: whether the 8 SES covariates controlled for in the main analysis differ systematically across districts (17) and estates (224).

- **Data**: raw survey data (pre-imputation) `merged_data0925.xlsx`
- **SES variables**: identical to the main-analysis `ses_vars` (Age, Gender, Edu, Emp, Imm, CD, DurRes, LivArea)
- **District level**: Kruskal-Wallis test (17 groups)
- **Estate level**: intraclass correlation coefficient ICC (224 estates)
- **Output**: `Output/01-Env-health/Table_S28_Balance_Test.csv`

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

RAW_DIR = "../../Data/Raw"
OUT_DIR = "../../Output/01-Env-health"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# -- Load raw data (pre-imputation) --------------------------------
df = pd.read_excel(f"{RAW_DIR}/merged_data0925.xlsx")
print(f"Raw sample size: {len(df)}")
print(f"Unique districts: {df['district'].nunique()}  Unique estates: {df['estate'].nunique()}")

In [ ]:
# -- Prepare SES variables (identical to the main-analysis ses_vars) --

# 1. LivArea: reproduce livareaperson = livarea_val / livnum_val from the R script
livnum_map  = {1:1, 2:2, 3:3, 4:4, 5:5, 6:6, 7:7.5}
livarea_map = {1:75, 2:175, 3:225, 4:275, 5:325, 6:375, 7:425,
               8:475, 9:575, 10:675, 11:775, 12:825, 13:875, 14:975, 15:1100}

def safe_map(val, mapping):
    try:
        return mapping[int(float(val))]
    except:
        return np.nan

df['livnum_val']  = df['livnum'].apply(lambda x: safe_map(x, livnum_map))
df['livarea_val'] = df['livarea'].apply(lambda x: safe_map(x, livarea_map))
df['LivArea']     = df['livarea_val'] / df['livnum_val']

# 2. DurRes: full reproduction of the R script case_when (year-by-year bins).
dura_map = {
    '1-2 年': 1,  '2-3 年': 1,  '3-4 年': 1,  '4-5 年': 1,
    '5-6年':  2,  '6-7年':  2,  '7-8年':  2,  '8-9年':  2,  '9-10年': 2,
    '10-11年': 3, '11-12年': 3, '12-13年': 3, '13-14年': 3, '14-15年': 3,
    '15-16年': 4, '16-17年': 4, '17-18年': 4, '18-19年': 4, '19-20年': 4,
    '超過20年': 5,
}
df['DurRes'] = df['liviDura'].map(dura_map)
still_na = df['DurRes'].isna()
df.loc[still_na, 'DurRes'] = pd.to_numeric(df.loc[still_na, 'liviDura'], errors='coerce')

# 3. CD (any disease): reproduce ifelse(disease == "1", 1, 0) from the R script.
disease_ch_map = {
    '沒有，完全健康':               1,
    '僅患有慢性疾病':               2,
    '僅患有精神疾病':               3,
    '同時患有慢性疾病和精神疾病':   4,
}
df['disease_num'] = df['disease'].map(disease_ch_map)
still_na = df['disease_num'].isna()
df.loc[still_na, 'disease_num'] = pd.to_numeric(df.loc[still_na, 'disease'], errors='coerce')
df['CD'] = (df['disease_num'] == 1).astype(float)
df.loc[df['disease_num'].isna(), 'CD'] = np.nan

# 4. Standardise column names
df = df.rename(columns={'age':'Age', 'gender':'Gender', 'edu':'Edu',
                         'emp':'Emp', 'imm':'Imm'})

# 5. Coerce all to numeric
SES_VARS = ['Age', 'Gender', 'Edu', 'Emp', 'Imm', 'CD', 'DurRes', 'LivArea']
for v in SES_VARS:
    df[v] = pd.to_numeric(df[v], errors='coerce')

# 6. Keep rows that have district / estate
df = df.dropna(subset=['district', 'estate']).copy()

print(f"Valid sample size: {len(df)}")
print(f"Districts: {df['district'].nunique()}   Estates: {df['estate'].nunique()}")
print("\nValid N and mean per variable:")
print(df[SES_VARS].agg(['count','mean','std']).round(2))

print(f"\nDurRes non-missing: {df['DurRes'].notna().sum()}")
print(f"CD distribution:\n{df['CD'].value_counts(dropna=False)}")

In [ ]:
# -- District level: Kruskal-Wallis test (17 groups) ---------------

def icc_one_way(series, groups):
    """One-way ANOVA ICC(1): between-group variance / total variance."""
    data = pd.DataFrame({'y': series, 'g': groups}).dropna()
    grand_mean = data['y'].mean()
    k = data['g'].nunique()
    n = len(data)
    group_sizes  = data.groupby('g')['y'].count()
    group_means  = data.groupby('g')['y'].mean()
    SS_between = sum(group_sizes[g] * (group_means[g] - grand_mean)**2 for g in group_means.index)
    SS_within  = sum(((data[data['g']==g]['y'] - group_means[g])**2).sum() for g in group_means.index)
    df_between = k - 1
    df_within  = n - k
    if df_within == 0 or df_between == 0:
        return np.nan
    MS_between = SS_between / df_between
    MS_within  = SS_within  / df_within
    n0 = (n - sum(s**2 for s in group_sizes) / n) / df_between
    icc = (MS_between - MS_within) / (MS_between + (n0 - 1) * MS_within)
    return max(icc, 0.0)

rows = []
for var in SES_VARS:
    col = df[var].dropna()
    n_valid = col.shape[0]
    overall_mean = col.mean()
    overall_sd   = col.std()

    # Kruskal-Wallis (district)
    district_groups = [grp[var].dropna().values
                       for _, grp in df.groupby('district')
                       if grp[var].dropna().shape[0] > 0]
    if len(district_groups) >= 2:
        try:
            kw_H, kw_p = stats.kruskal(*district_groups)
            kw_df = df['district'].nunique() - 1
        except ValueError:
            kw_H, kw_p, kw_df = np.nan, np.nan, np.nan
    else:
        kw_H, kw_p, kw_df = np.nan, np.nan, np.nan

    # ICC (estate)
    icc_val = icc_one_way(df[var], df['estate'])

    rows.append({
        'Variable'          : var,
        'N'                 : n_valid,
        'Overall Mean (SD)' : f"{overall_mean:.2f} ({overall_sd:.2f})",
        'KW H'              : round(kw_H, 2) if not np.isnan(kw_H) else np.nan,
        'df'                : int(kw_df) if not np.isnan(kw_df) else np.nan,
        'p (district)'      : round(kw_p, 3) if not np.isnan(kw_p) else np.nan,
        'ICC (estate)'      : round(icc_val, 3) if icc_val is not None else np.nan,
    })

df_result = pd.DataFrame(rows)

# Add significance label
def sig_label(p):
    if pd.isna(p): return ''
    if p < 0.001:  return '< 0.001'
    return f"{p:.3f}"

df_result['p (district)'] = df_result['p (district)'].apply(sig_label)

print(df_result.to_string(index=False))

In [ ]:
# -- Export Table S28 --------------------------------------------
out_path = f"{OUT_DIR}/Table_S28_Balance_Test.csv"
df_result.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Saved: {out_path}")

out_xlsx = f"{OUT_DIR}/Table_S28_Balance_Test.xlsx"
df_result.to_excel(out_xlsx, index=False)
print(f"Saved: {out_xlsx}")

In [ ]:
# -- Per-district SES means (auxiliary check) --------------------
district_summary = df.groupby('district')[SES_VARS].mean().round(2)
print(district_summary)